# summary_metadata_extraction_pt4
## Inventory Drift Calculator — Pivot Augmentation

This notebook:
1) Reads the Excel exports in `DatabaseExports/`
2) Filters *invalid artifact rows* **without normalizing** text
3) Classifies inventory events with drift-aware semantics (`inventory_type_class`, `inventory_cause`)
4) Builds an augmented Drake-style wide pivot (Yes/No + semantics per year)

**No text normalization is performed.** We only add row-validity flags and derived fields.


In [5]:
# ============================================
# Colab Bootstrap (Drive + Paths + File checks)
# ============================================

from google.colab import drive
from pathlib import Path

# 1) Mount Drive (safe if already mounted)
drive.mount("/content/drive")

# 2) Define project paths (EDIT ONLY THIS LINE if needed)
PROJECT_ROOT = Path("/content/drive/MyDrive/InventoryDriftCalculator")
EXPORTS_PATH = PROJECT_ROOT / "DatabaseExports"
OUTPUTS_PATH = PROJECT_ROOT / "Outputs"

# 3) Create Outputs folder if missing
OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)

# 4) Sanity checks
print("PROJECT_ROOT :", PROJECT_ROOT)
print("EXPORTS_PATH :", EXPORTS_PATH, "| exists =", EXPORTS_PATH.exists())
print("OUTPUTS_PATH :", OUTPUTS_PATH, "| exists =", OUTPUTS_PATH.exists())

# 5) Expected files (in DatabaseExports)
FILE_TREE_AGE = EXPORTS_PATH / "inventories_tree_age.xlsx"
FILE_META     = EXPORTS_PATH / "inventories_metadata.xlsx"
FILE_PIVOT    = EXPORTS_PATH / "inventory_history_pivot.xlsx"

print("\n=== Expected files ===")
for p in [FILE_TREE_AGE, FILE_META, FILE_PIVOT]:
    print(p.name, "->", p, "| exists =", p.exists())

# 6) Helpful: list xlsx in exports
print("\n=== XLSX files in DatabaseExports ===")
xlsx_files = sorted(EXPORTS_PATH.glob("*.xlsx"))
for f in xlsx_files:
    print("-", f.name)

# Optional: CD into exports for convenience (comment out if you prefer)
%cd $EXPORTS_PATH


Mounted at /content/drive
PROJECT_ROOT : /content/drive/MyDrive/InventoryDriftCalculator
EXPORTS_PATH : /content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports | exists = True
OUTPUTS_PATH : /content/drive/MyDrive/InventoryDriftCalculator/Outputs | exists = True

=== Expected files ===
inventories_tree_age.xlsx -> /content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports/inventories_tree_age.xlsx | exists = True
inventories_metadata.xlsx -> /content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports/inventories_metadata.xlsx | exists = True
inventory_history_pivot.xlsx -> /content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports/inventory_history_pivot.xlsx | exists = True

=== XLSX files in DatabaseExports ===
- MaindatabasevsIMC.xlsx
- NULL INSPECTION with maindatabase.xlsx
- Out_8.xlsx
- Result_11.xlsx
- Result_15.xlsx
- changelog.xlsx
- chunk_planeacion_cr.xlsx
- contract_replacements_ts.xlsx
- df_inventory_plan_2026.xlsx
- df_output.xlsx
- inventories_autoa

In [6]:
from pathlib import Path

# ✅ Ensure BASE_PATH is a Path (not str)
BASE_PATH = Path("/content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports")

# --- Inventory files (expected)
FILE_TREE_AGE = BASE_PATH / "inventories_tree_age.xlsx"
FILE_META     = BASE_PATH / "inventories_metadata.xlsx"
FILE_PIVOT    = BASE_PATH / "inventory_history_pivot.xlsx"

missing = [p for p in [FILE_TREE_AGE, FILE_META, FILE_PIVOT] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(str(x) for x in missing))

print("OK: files found")


OK: files found


In [11]:
# --- Load datasets
df_tree_age = pd.read_excel(FILE_TREE_AGE)
df_meta     = pd.read_excel(FILE_META)
df_pivot    = pd.read_excel(FILE_PIVOT)

print("tree_age rows:", len(df_tree_age), "cols:", df_tree_age.columns.tolist())
print("meta rows:    ", len(df_meta),     "cols:", df_meta.columns.tolist())
print("pivot rows:   ", len(df_pivot),    "cols:", df_pivot.columns.tolist())


tree_age rows: 382 cols: ['Unnamed: 0', 'contract_code', 'inventory_year', 'planting_year', 'inventory_type', 'tree_age', 'status']
meta rows:     204 cols: ['Unnamed: 0', 'contract_code', 'inventory_type', 'farmer_name', 'cruiser_name', 'region', 'planting_year', 'source_file', 'inventory_date', 'inventory_year']
pivot rows:    453 cols: ['contract_code', 'contract_name', 'farmer_number', 'status', 'planting_year', 2021, 2022, 2023, 2024, 2025]


In [12]:
# --- Quick peek (safe, no edits)
display(df_tree_age.head(5))
display(df_meta.head(5))
display(df_pivot.head(5))


,Unnamed: 0,contract_code,inventory_year,planting_year,inventory_type,tree_age,status
0,0,CR0013,2021.0,2017.0,Permanent,4.0,Out of Program
1,1,CR0028,2021.0,2018.0,Inventory,3.0,Out of Program
2,2,CR0029,2021.0,2018.0,Inventory,3.0,Out of Program
3,3,CR0030,2021.0,2018.0,Inventory,3.0,Active
4,4,CR0035,2021.0,2020.0,Survival1,1.0,Active


,Unnamed: 0,contract_code,inventory_type,farmer_name,cruiser_name,region,planting_year,source_file,inventory_date,inventory_year
0,0,20078-CR0086-I1,RFT,Edgar Chinchilla Quesada,Alejandro Quesada and Mariana Alfaro,Costa Rica,2019.0,CR0086_2019_Agner Chinchila.xlsx,NaN,NaN
1,1,20031-CR0035-I1,Inventory,Alex Donato Arguedas Arroyo,Mariana Alfaro y Alejandro Quesada,Costa Rica,2020.0,CR0035_2020_Donato Arguedas.xlsx,2024-02-09,2024.0
2,2,20033-CR0037-I1,Inventory,Hermanos Wolfe Arce S.A,Mariana y Alejandro,Costa Rica,2020.0,CR0037 - 2023 - Ana Wolfe Arce.xlsx,NaN,NaN
3,3,20069-CR0075-I1,Permanent,Oldemar Perez,Gustavo Solis Chavarria y Greivin Hernandez,Costa Rica,2021.0,CR0092_2023_Christian Araya.xlsx,2024-01-23,2024.0
4,4,20091-CR0099-I1,Survival1,Jose Francisco Angulo,Alejandro,Costa Rica,2022.0,CR0099_2023_Corporacion Comercial San Miguel d...,2024-02-07,2024.0


,contract_code,contract_name,farmer_number,status,planting_year,2021,2022,2023,2024,2025
0,CR0001,Cecilia Castillo Umaña,20001.0,Out of Program,2016,NaN,NaN,NaN,NaN,NaN
1,CR0002,Coopedota,20002.0,Out of Program,2016,NaN,NaN,NaN,NaN,NaN
2,CR0003,Agricola Los Robles de Naranjo S.A,20003.0,Out of Program,2016,NaN,NaN,NaN,NaN,NaN
3,CR0004,Fernando Esteban Chacon,20004.0,Out of Program,2016,NaN,NaN,NaN,NaN,NaN
4,CR0005,"ASESORIA DE EMPRESAS TONAL S A,",20005.0,Out of Program,2016,NaN,NaN,NaN,NaN,NaN


## 1) Row validity (NO normalization)
We do **not** trim/casefold/replace any strings.
We only flag rows that are clearly non-data artifacts (e.g., report headers like `Árboles Contratados:` or numeric-only strings in `inventory_type`).


In [13]:
# --- Define explicit row-validity rules for inventories_tree_age

# Allowed raw inventory_type values (exact, case-sensitive) observed so far.
# IMPORTANT: We are NOT normalizing; we list accepted raw values explicitly.
ALLOWED_INVENTORY_TYPE_RAW = set([
    "Permanent",
    "Inventory",
    "Survival1",
    "Survival0",
    "RFT",
    "Survival 1",
    "Survival 0",
    "inventory",
])

def is_numeric_only(x):
    try:
        s = str(x)
    except Exception:
        return False
    return bool(re.fullmatch(r"[0-9]+", s.strip()))

def is_artifact_inventory_type(x):
    # Explicit known artifact patterns (no normalization, just detection)
    if pd.isna(x):
        return True
    s = str(x)
    if s == "Árboles Contratados:":
        return True
    if is_numeric_only(s):
        return True
    return False

def row_valid_tree_age(r):
    # Require core fields present
    if pd.isna(r.get("contract_code")):
        return False
    # inventory_year / planting_year / tree_age should be numeric
    for k in ["inventory_year", "planting_year", "tree_age"]:
        if pd.isna(r.get(k)):
            return False
        # allow ints/floats
        try:
            float(r.get(k))
        except Exception:
            return False
    inv_type = r.get("inventory_type")
    if is_artifact_inventory_type(inv_type):
        return False
    # Keep only inventory_type values we explicitly accept
    if str(inv_type) not in ALLOWED_INVENTORY_TYPE_RAW:
        # Unknown values are treated as invalid for drift logic (but preserved in raw file)
        return False
    return True

df_tree_age["row_valid"] = df_tree_age.apply(row_valid_tree_age, axis=1)

print("Valid rows:", df_tree_age["row_valid"].sum(), "/", len(df_tree_age))
print("Invalid inventory_type examples:")
display(df_tree_age.loc[~df_tree_age["row_valid"], ["contract_code","inventory_year","planting_year","tree_age","inventory_type"]].head(20))


Valid rows: 348 / 382
Invalid inventory_type examples:


,contract_code,inventory_year,planting_year,tree_age,inventory_type
175,CR0086,NaN,2019.0,NaN,RFT
177,CR0037,NaN,2020.0,NaN,Inventory
191,CR0105,NaN,2023.0,NaN,Survival0
192,CR0101,NaN,2023.0,NaN,Survival0
195,CR0101,NaN,2023.0,NaN,Survival0
197,CR0058,NaN,NaN,NaN,Árboles Contratados:
212,CR0100,NaN,2023.0,NaN,Survival0
223,GT0044,NaN,2023.0,NaN,Survival0
228,GT0043,NaN,2022.0,NaN,Survival1
229,GT0041,NaN,2022.0,NaN,Survival1


## 2) Drift-aware classification
We classify **inventory events** based on `tree_age` (age at inventory year), not on raw `inventory_type` text.

Baseline bins:
- Inventory_1: ages in [2, 4] (planned=3, drift ±1)
- Inventory_2: ages in [5, 7] (planned=6, drift ±1)
- Inventory_3: ages in [8, 10] (planned=9, drift ±1)

`inventory_cause` is computed relative to the planned age (3/6/9): on-time / early / late / outside.


In [14]:
# --- Drift windows (explicit, configurable)
# NOTE: these are *classification windows*, not normalization rules.
WINDOWS = [
    {"inv":"Inventory_1", "planned_age":3, "min_age":2, "max_age":4},
    {"inv":"Inventory_2", "planned_age":6, "min_age":5, "max_age":7},
    {"inv":"Inventory_3", "planned_age":9, "min_age":8, "max_age":10},
]

def classify_inventory_by_age(age):
    # returns (inventory_type_class, planned_age) or (None, None)
    for w in WINDOWS:
        if w["min_age"] <= age <= w["max_age"]:
            return w["inv"], w["planned_age"]
    return None, None

def cause_from_age(age, planned_age):
    if planned_age is None:
        return "Outside_inventory_window"
    if age == planned_age:
        return "Executed_on_planned_year"
    if age < planned_age:
        # age is within window but before planned age
        return "Executed_early_drift"
    if age > planned_age:
        return "Executed_late_drift"
    return "Outside_inventory_window"

df_events = df_tree_age[df_tree_age["row_valid"]].copy()

# Ensure numeric
df_events["tree_age_num"] = pd.to_numeric(df_events["tree_age"], errors="coerce")

df_events[["inventory_type_class","planned_age"]] = df_events["tree_age_num"].apply(
    lambda a: pd.Series(classify_inventory_by_age(a))
)

df_events["inventory_cause"] = df_events.apply(
    lambda r: cause_from_age(r["tree_age_num"], r["planned_age"]),
    axis=1
)

# Keep minimal event-level fields
keep_cols = [
    "contract_code","inventory_year","planting_year","tree_age","tree_age_num",
    "inventory_type", "status",
    "inventory_type_class","inventory_cause"
]
df_events = df_events[keep_cols].copy()

print("Event-level rows (valid):", len(df_events))
display(df_events.head(20))


Event-level rows (valid): 348


,contract_code,inventory_year,planting_year,tree_age,tree_age_num,inventory_type,status,inventory_type_class,inventory_cause
0,CR0013,2021.0,2017.0,4.0,4.0,Permanent,Out of Program,Inventory_1,Executed_late_drift
1,CR0028,2021.0,2018.0,3.0,3.0,Inventory,Out of Program,Inventory_1,Executed_on_planned_year
2,CR0029,2021.0,2018.0,3.0,3.0,Inventory,Out of Program,Inventory_1,Executed_on_planned_year
3,CR0030,2021.0,2018.0,3.0,3.0,Inventory,Active,Inventory_1,Executed_on_planned_year
4,CR0035,2021.0,2020.0,1.0,1.0,Survival1,Active,None,Outside_inventory_window
5,CR0046,2021.0,2020.0,1.0,1.0,Survival1,Active,None,Outside_inventory_window
6,CR0052,2021.0,2020.0,1.0,1.0,Survival1,Active,None,Outside_inventory_window
7,CR0057,2021.0,2020.0,1.0,1.0,Survival1,Out of Program,None,Outside_inventory_window
8,CR0058,2021.0,2020.0,1.0,1.0,Survival1,Out of Program,None,Outside_inventory_window
9,CR0059,2021.0,2020.0,1.0,1.0,Survival1,Active,None,Outside_inventory_window


## 3) Enrich events with `inventories_metadata.xlsx` (optional join)
We join on `(contract_code, inventory_year)` to bring `inventory_date`, `source_file`, etc.
This is best-effort and **does not overwrite** drift classification.


In [15]:
# --- Best-effort join with metadata
# NOTE: metadata inventory_type may differ from tree_age inventory_type; we keep both.
df_meta_small = df_meta.copy()

# Keep only columns we care about, if present
meta_cols = [c for c in [
    "contract_code","inventory_year","inventory_type","inventory_date","source_file","region","cruiser_name","farmer_name"
] if c in df_meta_small.columns]
df_meta_small = df_meta_small[meta_cols].copy()

df_events_enriched = df_events.merge(
    df_meta_small,
    on=["contract_code","inventory_year"],
    how="left",
    suffixes=("_tree_age","_meta")
)

display(df_events_enriched.head(20))


,contract_code,inventory_year,planting_year,tree_age,tree_age_num,inventory_type_tree_age,status,inventory_type_class,inventory_cause,inventory_type_meta,inventory_date,source_file,region,cruiser_name,farmer_name
0,CR0013,2021.0,2017.0,4.0,4.0,Permanent,Out of Program,Inventory_1,Executed_late_drift,NaN,NaN,NaN,NaN,NaN,NaN
1,CR0028,2021.0,2018.0,3.0,3.0,Inventory,Out of Program,Inventory_1,Executed_on_planned_year,NaN,NaN,NaN,NaN,NaN,NaN
2,CR0029,2021.0,2018.0,3.0,3.0,Inventory,Out of Program,Inventory_1,Executed_on_planned_year,NaN,NaN,NaN,NaN,NaN,NaN
3,CR0030,2021.0,2018.0,3.0,3.0,Inventory,Active,Inventory_1,Executed_on_planned_year,NaN,NaN,NaN,NaN,NaN,NaN
4,CR0035,2021.0,2020.0,1.0,1.0,Survival1,Active,None,Outside_inventory_window,NaN,NaN,NaN,NaN,NaN,NaN
5,CR0046,2021.0,2020.0,1.0,1.0,Survival1,Active,None,Outside_inventory_window,NaN,NaN,NaN,NaN,NaN,NaN
6,CR0052,2021.0,2020.0,1.0,1.0,Survival1,Active,None,Outside_inventory_window,NaN,NaN,NaN,NaN,NaN,NaN
7,CR0057,2021.0,2020.0,1.0,1.0,Survival1,Out of Program,None,Outside_inventory_window,NaN,NaN,NaN,NaN,NaN,NaN
8,CR0058,2021.0,2020.0,1.0,1.0,Survival1,Out of Program,None,Outside_inventory_window,NaN,NaN,NaN,NaN,NaN,NaN
9,CR0059,2021.0,2020.0,1.0,1.0,Survival1,Active,None,Outside_inventory_window,NaN,NaN,NaN,NaN,NaN,NaN


## 4) Build an augmented Drake pivot
We keep the existing wide Yes/No structure and add:
- `inventory_type_<YEAR>`
- `inventory_cause_<YEAR>`

We do **not** normalize years; we only process the year columns that already exist in Drake's pivot.


In [ ]:
# --- Identify year columns in Drake pivot (int years like 2021, 2022, ...)
base_cols = ["contract_code","contract_name","farmer_number","status","planting_year"]
year_cols = [c for c in df_pivot.columns if isinstance(c, (int, float)) and str(int(c)).isdigit()]
year_cols = sorted([int(c) for c in year_cols])

print("Year columns detected:", year_cols)

# Create a lookup from events: for each (contract_code, inventory_year) choose the "best" classification if duplicates exist.
# Priority: on_planned > early/late drift > outside
CAUSE_PRIORITY = {
    "Executed_on_planned_year": 3,
    "Executed_early_drift": 2,
    "Executed_late_drift": 2,
    "Outside_inventory_window": 1,
    "Not_executed_yet_in_window": 0,
    "Already_executed_nearby_year": 0,
}

tmp = df_events_enriched.copy()
tmp["cause_priority"] = tmp["inventory_cause"].map(CAUSE_PRIORITY).fillna(0)

# If multiple rows per contract/year exist, keep the highest priority row
tmp = tmp.sort_values(["contract_code","inventory_year","cause_priority"], ascending=[True, True, False])
tmp_best = tmp.drop_duplicates(subset=["contract_code","inventory_year"], keep="first")

lookup = tmp_best.set_index(["contract_code","inventory_year"])[["inventory_type_class","inventory_cause"]]

df_pivot_aug = df_pivot.copy()

for y in year_cols:
    type_col = f"inventory_type_{y}"
    cause_col = f"inventory_cause_{y}"
    # Map from lookup
    df_pivot_aug[type_col] = df_pivot_aug["contract_code"].apply(
        lambda cc: lookup.loc[(cc, y), "inventory_type_class"] if (cc, y) in lookup.index else None
    )
    df_pivot_aug[cause_col] = df_pivot_aug["contract_code"].apply(
        lambda cc: lookup.loc[(cc, y), "inventory_cause"] if (cc, y) in lookup.index else None
    )

display(df_pivot_aug.head(10))


## 5) Save outputs
- `inventory_events_normalized.xlsx`: event-level table with drift semantics
- `inventory_history_pivot_augmented.xlsx`: Drake pivot + semantic columns


In [ ]:
OUT_EVENTS = BASE_PATH / "inventory_events_normalized.xlsx"
OUT_PIVOT  = BASE_PATH / "inventory_history_pivot_augmented.xlsx"

df_events_enriched.to_excel(OUT_EVENTS, index=False)
df_pivot_aug.to_excel(OUT_PIVOT, index=False)

print("Saved:")
print(" -", OUT_EVENTS)
print(" -", OUT_PIVOT)


## Notes / Decisions captured
- No text normalization.
- Only explicit artifact filtering for `inventories_tree_age.xlsx`.
- Drift windows are configurable via `WINDOWS`.
- Pivot augmentation uses only the year columns already present in `inventory_history_pivot.xlsx`.


##MID escribió el siguiente código con el que encontramos falta de trazabilidad que está afectando el join

In [4]:
import pandas as pd
import re
from pathlib import Path

EXPORTS = Path("/content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports")

df_tree = pd.read_excel(EXPORTS / "inventories_tree_age.xlsx")
df_meta = pd.read_excel(EXPORTS / "inventories_metadata.xlsx")

# ----------------------------
# 1) Build contract_code_key (derived, originals untouched)
# ----------------------------
# tree: already clean
df_tree["contract_code_key"] = df_tree["contract_code"]

# meta: extract CR#### from strings like "20031-CR0035-I1"
df_meta["contract_code_key"] = df_meta["contract_code"].astype(str).str.extract(r"(CR\d{4})", expand=False)

# quick sanity check
print("Meta contract_code_key null rate:", df_meta["contract_code_key"].isna().mean())
print(df_meta[["contract_code", "contract_code_key"]].head(10))

# ----------------------------
# 2) Build inventory_year_key (derived)
# ----------------------------
# tree: numeric year
df_tree["inventory_year_key"] = pd.to_numeric(df_tree["inventory_year"], errors="coerce").astype("Int64")

# meta: prefer inventory_year if present; else derive from inventory_date; else derive from source_file
meta_year = pd.to_numeric(df_meta["inventory_year"], errors="coerce")

# from inventory_date (YYYY-MM-DD)
meta_year_from_date = pd.to_datetime(df_meta["inventory_date"], errors="coerce").dt.year

# from source_file (pick first 4-digit year 19xx/20xx)
meta_year_from_source = df_meta["source_file"].astype(str).str.extract(r"((?:19|20)\d{2})", expand=False)
meta_year_from_source = pd.to_numeric(meta_year_from_source, errors="coerce")

df_meta["inventory_year_key"] = (
    meta_year.fillna(meta_year_from_date).fillna(meta_year_from_source)
).astype("Int64")

print("Meta inventory_year_key null rate:", df_meta["inventory_year_key"].isna().mean())
print(df_meta[["inventory_year", "inventory_date", "source_file", "inventory_year_key"]].head(10))

# ----------------------------
# 3) FILTER obvious non-data artifacts in tree_age (explicit only)
# ----------------------------
GARBAGE_TYPES = {"Árboles Contratados:", "12", "0"}
df_tree_valid = df_tree[
    df_tree["contract_code_key"].notna() &
    df_tree["inventory_year_key"].notna() &
    (~df_tree["inventory_type"].isin(GARBAGE_TYPES))
].copy()

# ----------------------------
# 4) Aggregate metadata by (contract_code_key, inventory_year_key) to avoid multi-row explosions
# ----------------------------
meta_cols = ["contract_code_key","inventory_year_key","inventory_type","inventory_date","source_file","region","cruiser_name","farmer_name"]
df_meta_slim = df_meta[meta_cols].copy()

def first_non_null(s):
    s2 = s.dropna()
    return s2.iloc[0] if len(s2) else pd.NA

df_meta_agg = df_meta_slim.groupby(["contract_code_key","inventory_year_key"], as_index=False).agg({
    "inventory_type": first_non_null,
    "inventory_date": first_non_null,
    "source_file": first_non_null,
    "region": first_non_null,
    "cruiser_name": first_non_null,
    "farmer_name": first_non_null
}).rename(columns={"inventory_type": "inventory_type_meta"})

# ----------------------------
# 5) Join tree_age -> meta (NOW it should match)
# ----------------------------
df_join = df_tree_valid.merge(
    df_meta_agg,
    on=["contract_code_key","inventory_year_key"],
    how="left"
)

match_rate = df_join["inventory_type_meta"].notna().mean()
print(f"\n✅ Meta match rate after key-derivation: {match_rate:.1%}")

print("\nSample matched rows:")
print(df_join[df_join["inventory_type_meta"].notna()][
    ["contract_code","inventory_year_key","inventory_type","inventory_type_meta","inventory_date","source_file"]
].head(10))

print("\nSample still-missing meta rows (to debug):")
print(df_join[df_join["inventory_type_meta"].isna()][
    ["contract_code","inventory_year_key","inventory_type"]
].head(10))


Meta contract_code_key null rate: 0.7352941176470589
     contract_code contract_code_key
0  20078-CR0086-I1            CR0086
1  20031-CR0035-I1            CR0035
2  20033-CR0037-I1            CR0037
3  20069-CR0075-I1            CR0075
4  20091-CR0099-I1            CR0099
5  20038-CR0042-I1            CR0042
6  20040-CR0044-I1            CR0044
7  20096-CR0104-I1            CR0104
8  20042-CR0046-I1            CR0046
9  20079-CR0087-I1            CR0087
Meta inventory_year_key null rate: 0.03431372549019608
   inventory_year inventory_date  \
0             NaN            NaN   
1          2024.0     2024-02-09   
2             NaN            NaN   
3          2024.0     2024-01-23   
4          2024.0     2024-02-07   
5          2024.0     2024-01-31   
6          2024.0     2024-11-01   
7          2024.0     2024-02-02   
8          2024.0     2024-01-15   
9          2024.0     2024-02-23   

                                         source_file  inventory_year_key  
0            

In [7]:
# PT4 — Advanced 2026 (MINIMAL OUTPUT + DUE vs OVERDUE)
from __future__ import annotations

import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

BASE_YEAR = 2026

tree_path = Path("/content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports/inventories_tree_age.xlsx")
pivot_path = Path("/content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports/inventory_history_pivot.xlsx")
out_path = Path("/content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports/pt4_inventory_advanced_2026.xlsx")

INCLUDE_AUDIT_SHEETS = False  # deja False


# ----------------------------
# Helpers
# ----------------------------
def clean_inventory_type(raw) -> str:
    if pd.isna(raw):
        return ""
    s = str(raw).strip()
    if s == "0" or s.lower() == "no":
        return "NO"
    s_norm = re.sub(r"\s+", "", s).lower()
    mapping = {
        "survival0": "Survival0",
        "survival1": "Survival1",
        "survival2": "Survival2",
        "inventory": "Inventory",
        "permanent": "Permanent",
        "permanentplot": "Permanent",
        "possibleharvest": "PossibleHarvest",
        "rft": "RFT",
    }
    return mapping.get(s_norm, s)


def expected_type_from_tree_age(age: float) -> str:
    if pd.isna(age):
        return ""
    a = int(age)
    if a in (0, 1):
        return f"Survival{a}"
    if a == 3:
        return "Inventory1"
    if a == 6:
        return "Inventory2"
    if a == 9:
        return "Inventory3"
    if a in (2, 4, 5, 7, 8):
        return "Permanent"
    if a in (10, 11):
        return "PossibleHarvest"
    return ""


def inventory_level_from_tree_age(age: float) -> str:
    if pd.isna(age):
        return ""
    if age < 6:
        return "Inventory1"
    if age < 9:
        return "Inventory2"
    return "Inventory3"


PRIORITY = {
    "Inventory3": 100,
    "Inventory2": 90,
    "Inventory1": 80,
    "Inventory": 70,
    "Permanent": 60,
    "PossibleHarvest": 50,
    "Survival2": 40,
    "Survival1": 30,
    "Survival0": 20,
    "NO": 0,
    "": -1,
}


def normalize_event_type(row: pd.Series) -> str:
    it = row.get("inventory_type_clean", "")
    age = row.get("tree_age", np.nan)
    if it == "Inventory":
        return inventory_level_from_tree_age(age)
    if it in PRIORITY:
        return it
    derived = expected_type_from_tree_age(age)
    return derived or it


def is_title_row(row: pd.Series) -> bool:
    cc = str(row.get("contract_code", "")).strip()
    it_raw = str(row.get("inventory_type", "")).strip().lower()
    if cc == "" or cc.lower().startswith("nan"):
        return True
    if "árboles" in it_raw or "arboles" in it_raw:
        return True
    if pd.isna(row.get("inventory_year", np.nan)) and pd.isna(row.get("planting_year", np.nan)):
        return True
    return False


def status_is_out(status_val) -> bool:
    if pd.isna(status_val):
        return True
    s = str(status_val).strip().lower()
    return ("rft" in s) or ("out of program" in s) or ("outofprogram" in s)


def status_is_pending(status_val) -> bool:
    if pd.isna(status_val):
        return False
    return str(status_val).strip().lower().startswith("pending")


def status_is_active(status_val) -> bool:
    if pd.isna(status_val):
        return False
    return str(status_val).strip().lower().startswith("active")


def drift(li, due) -> Optional[int]:
    if pd.isna(due):
        return None
    due_i = int(due)
    if pd.isna(li):
        # 0 => due now, >0 => overdue now
        return int(BASE_YEAR - due_i)
    # negative => early, positive => late
    return int(int(li) - due_i)


def due_bucket(current_year: int, due_year) -> str:
    if due_year is None or pd.isna(due_year):
        return "UNKNOWN"
    dy = int(due_year)
    if current_year < dy:
        return "NOT_DUE"
    if current_year == dy:
        return "DUE"
    return "OVERDUE"


def completion_state(last_done, current_year: int, due_year) -> str:
    if not pd.isna(last_done):
        return "DONE"
    return due_bucket(current_year, due_year)


# ----------------------------
# Load
# ----------------------------
df_tree = pd.read_excel(tree_path)
df_pivot = pd.read_excel(pivot_path)

# ----------------------------
# Clean events
# ----------------------------
df_tree = df_tree.drop(
    columns=[c for c in df_tree.columns if str(c).lower().startswith("unnamed")],
    errors="ignore",
).copy()

df_tree["inventory_type_clean"] = df_tree["inventory_type"].map(clean_inventory_type)
df_tree["event_type"] = df_tree.apply(normalize_event_type, axis=1)

mask_title = df_tree.apply(is_title_row, axis=1)
df_title_rows = df_tree[mask_title].copy()
df_tree = df_tree[~mask_title].copy()

df_tree["is_out"] = df_tree["status"].map(status_is_out)
df_tree["is_active"] = df_tree["status"].map(status_is_active)
df_tree["is_pending"] = df_tree["status"].map(status_is_pending)

# Only Active counts as "already done" (avoid pending weirdness)
df_done = df_tree[(~df_tree["is_out"]) & (df_tree["is_active"])].copy()

# Inventory wins per contract-year
df_done["priority"] = df_done["event_type"].map(lambda x: PRIORITY.get(x, 10))
df_done["inventory_year_int"] = pd.to_numeric(df_done["inventory_year"], errors="coerce").astype("Int64")
df_best = (
    df_done.sort_values(["contract_code", "inventory_year_int", "priority"], ascending=[True, True, False])
    .drop_duplicates(subset=["contract_code", "inventory_year_int"], keep="first")
    .copy()
)

# Typed history for 2021-2025
hist_years = [2021, 2022, 2023, 2024, 2025]
df_hist = df_best[df_best["inventory_year_int"].isin(hist_years)].copy()
pivot_typed = df_hist.pivot(index="contract_code", columns="inventory_year_int", values="event_type").reset_index()
pivot_typed = pivot_typed.rename(columns={y: f"event_type_{y}" for y in hist_years if y in pivot_typed.columns})


def last_year(event: str) -> pd.Series:
    return df_best.loc[df_best["event_type"] == event].groupby("contract_code")["inventory_year_int"].max()


last_inv1 = last_year("Inventory1")
last_inv2 = last_year("Inventory2")
last_inv3 = last_year("Inventory3")

# ----------------------------
# Base contracts + merge
# ----------------------------
base = df_pivot.copy()
base["planting_year_int"] = pd.to_numeric(base["planting_year"], errors="coerce").astype("Int64")
base["is_out"] = base["status"].map(status_is_out)
base["is_pending"] = base["status"].map(status_is_pending)

out = base.merge(pivot_typed, on="contract_code", how="left")
out["last_inventory1_year"] = out["contract_code"].map(last_inv1).astype("Int64")
out["last_inventory2_year"] = out["contract_code"].map(last_inv2).astype("Int64")
out["last_inventory3_year"] = out["contract_code"].map(last_inv3).astype("Int64")

out["inv1_due_year"] = (out["planting_year_int"] + 3).astype("Int64")
out["inv2_due_year"] = (out["planting_year_int"] + 6).astype("Int64")
out["inv3_due_year"] = (out["planting_year_int"] + 9).astype("Int64")

# ✅ DUE vs OVERDUE
out["inv1_due_bucket_2026"] = out["inv1_due_year"].map(lambda x: due_bucket(BASE_YEAR, x))
out["inv2_due_bucket_2026"] = out["inv2_due_year"].map(lambda x: due_bucket(BASE_YEAR, x))
out["inv3_due_bucket_2026"] = out["inv3_due_year"].map(lambda x: due_bucket(BASE_YEAR, x))

out["inv1_state_2026"] = out.apply(lambda r: completion_state(r["last_inventory1_year"], BASE_YEAR, r["inv1_due_year"]), axis=1)
out["inv2_state_2026"] = out.apply(lambda r: completion_state(r["last_inventory2_year"], BASE_YEAR, r["inv2_due_year"]), axis=1)
out["inv3_state_2026"] = out.apply(lambda r: completion_state(r["last_inventory3_year"], BASE_YEAR, r["inv3_due_year"]), axis=1)


def recommend_2026(row: pd.Series) -> tuple[str, str]:
    if bool(row.get("is_out", False)):
        return ("NO_ACTION", "OUT_OF_SCOPE")

    py = row.get("planting_year_int", pd.NA)

    # Pending: accept false positive
    if bool(row.get("is_pending", False)) and (pd.isna(py) or int(py) >= BASE_YEAR):
        return ("Survival0", "PENDING_ASSUME_SURV0")

    if pd.isna(py):
        return ("UNKNOWN", "MISSING_PLANTING_YEAR")

    py = int(py)
    li1, li2, li3 = row.get("last_inventory1_year"), row.get("last_inventory2_year"), row.get("last_inventory3_year")
    due1, due2, due3 = py + 3, py + 6, py + 9

    # ✅ reasons are DUE_* vs OVERDUE_* (no más "DUE_OR_OVERDUE")
    if pd.isna(li1) and BASE_YEAR >= due1:
        return ("Inventory1", f"{due_bucket(BASE_YEAR, due1)}_INV1")
    if (not pd.isna(li1)) and pd.isna(li2) and BASE_YEAR >= due2:
        return ("Inventory2", f"{due_bucket(BASE_YEAR, due2)}_INV2")
    if (not pd.isna(li2)) and pd.isna(li3) and BASE_YEAR >= due3:
        return ("Inventory3", f"{due_bucket(BASE_YEAR, due3)}_INV3")

    # Not due => wildcard fill
    age = BASE_YEAR - py
    fallback = expected_type_from_tree_age(age)
    return (fallback or "NO_ACTION", "AGE_FALLBACK")


out[["recommended_2026", "recommended_reason_2026"]] = out.apply(lambda r: pd.Series(recommend_2026(r)), axis=1)

out["age_2026"] = (BASE_YEAR - out["planting_year_int"].astype("float")).round(0).astype("Int64")
out["inv1_drift_years"] = out.apply(lambda r: drift(r["last_inventory1_year"], r["inv1_due_year"]), axis=1)
out["inv2_drift_years"] = out.apply(lambda r: drift(r["last_inventory2_year"], r["inv2_due_year"]), axis=1)
out["inv3_drift_years"] = out.apply(lambda r: drift(r["last_inventory3_year"], r["inv3_due_year"]), axis=1)

# ✅ Export includes due/state columns
final_cols = [
    "contract_code", "contract_name", "farmer_number", "status",
    "planting_year", "age_2026",
    "recommended_2026", "recommended_reason_2026",
    "last_inventory1_year", "last_inventory2_year", "last_inventory3_year",
    "inv1_due_year", "inv2_due_year", "inv3_due_year",
    "inv1_due_bucket_2026", "inv2_due_bucket_2026", "inv3_due_bucket_2026",
    "inv1_state_2026", "inv2_state_2026", "inv3_state_2026",
    "inv1_drift_years", "inv2_drift_years", "inv3_drift_years",
] + [f"event_type_{y}" for y in hist_years]

final_cols = [c for c in final_cols if c in out.columns]
out_final = out[final_cols].copy()

# Optional audit
df_dedup_candidates = pd.DataFrame()
if INCLUDE_AUDIT_SHEETS:
    keys = ["contract_code", "inventory_year_int", "event_type"]
    dup_mask = df_best.duplicated(subset=keys, keep=False)
    df_dedup_candidates = df_best[dup_mask].sort_values(keys).copy()

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    out_final.to_excel(writer, sheet_name="inventory_advanced_2026", index=False)
    if INCLUDE_AUDIT_SHEETS:
        df_title_rows.to_excel(writer, sheet_name="title_rows", index=False)
        df_dedup_candidates.to_excel(writer, sheet_name="dedup_candidates", index=False)

print("Wrote:", out_path)


Wrote: /content/drive/MyDrive/InventoryDriftCalculator/DatabaseExports/pt4_inventory_advanced_2026.xlsx
